In [1]:
import numpy as np
import pandas as pd
import cvxpy as cp
import seaborn as sns
import mosek
import matplotlib.pyplot as plt
import datetime as date
from datetime import datetime as dt
from dateutil.relativedelta import *
import scipy.stats
from scipy.stats import rankdata

In [4]:
def phi_calc(q,p):
    N = len(p)
    phi = 0
    for i in range(N):
        if q[i]<= 0:
            phi = phi + 0
        else:
            phi = phi + q[i]*np.log(q[i]/p[i])
    return(phi)

def direction(m):
    theta = np.random.normal(0,1,size = m)
    return(theta/np.sum(theta))

def L_maxi(p,q_0,theta,r):
    l_b = 20
    l_a = 0
    l = (l_b+ l_a)/2
    m = len(p)
    while l_b-l_a > 1e-5:
        if np.min(q_0 + l*theta)< -1e-10 or 1-np.sum(q_0+l*theta)<1e-10 or \
        phi_calc(np.concatenate((q_0+l*theta,[1-np.sum(q_0+l*theta)])),p) > r:
            l_b = l
            l = (l_b+l_a)/2
        else:
            l_a = l
            l = (l_b+l_a)/2
    return(l_a)
def L_mini (p,q_0,theta,r):
    l_b = 0
    l_a = -20
    l = (l_b+ l_a)/2
    m = len(p)
    while l_b-l_a > 1e-5:
        if np.min(q_0 + l*theta)< -1e-10 or 1-np.sum(q_0+l*theta)<1e-10 or \
        phi_calc(np.concatenate((q_0+l*theta,[1-np.sum(q_0+l*theta)])),p) > r:
            l_a = l
            l = (l_b+l_a)/2
        else:
            l_b = l
            l = (l_b+l_a)/2
    return(l_b)

def hit_and_run(p,phi_func,r,par,steps):
    N = len(p)
    q_0 = p[0:N-1]
    points = []
    for i in range(steps):
        theta = direction(N-1)
        l_max = L_maxi(p,q_0,theta,r)
        l_min = L_mini (p,q_0,theta,r)
        l_rand = np.random.uniform(l_min,l_max)
        q_new = np.concatenate((q_0+l_rand*theta, [1-np.sum(q_0+l_rand*theta)]))
        points.append(q_new)
        q_0 = q_new[0:N-1]
    return(points)